# Adverse Event Signal Exploration

**Purpose:** Explore de-identified/synthetic adverse-event reports for reporting patterns and potential signals requiring expert review.

> **Safety note:** This is a cautious descriptive analysis. A reporting signal is **not proof of causation**. The dataset is synthetic and aggregate-style; results must not be used for clinical decisions or to conclude that a drug caused an adverse event.


## 1. Objectives

- Clean drug, reaction, severity, age group, and outcome fields.
- Calculate reporting frequency and serious-report proportions.
- Visualize high-frequency drug–event combinations.
- Document limitations and identify patterns that could merit expert review.


In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom pathlib import Path\n\nplt.rcParams['figure.figsize'] = (8, 5)\nDATA_PATH = Path('adverse_drug_reactions_sample.csv')\n\ndf = pd.read_csv(DATA_PATH)\nprint('Shape:', df.shape)\ndisplay(df.head())

## 2. Data quality check


In [ ]:
print('Missing values before cleaning:')\ndisplay(df.isna().sum().to_frame('MissingCount'))\n\nprint('Duplicate rows:', df.duplicated().sum())\nprint('Columns:', list(df.columns))

## 3. Clean and standardize fields

The supplied file contains `PatientAge` rather than an explicit age-group column, so an `AgeGroup` field is derived for the required analysis.

In [ ]:
text_cols = ['ReportID', 'PatientGender', 'SuspectedDrug', 'Indication',\n             'AdverseReaction', 'Severity', 'Outcome', 'ReporterType']\nfor col in text_cols:\n    df[col] = df[col].astype('string').str.strip()\n\ndf['PatientAge'] = pd.to_numeric(df['PatientAge'], errors='coerce')\ndf['ReactionOnsetDays'] = pd.to_numeric(df['ReactionOnsetDays'], errors='coerce')\n\ndef age_group(age):\n    if pd.isna(age): return 'Unknown'\n    if age < 18: return 'Under 18'\n    if age <= 44: return '18–44'\n    if age <= 64: return '45–64'\n    return '65+'\n\ndf['AgeGroup'] = df['PatientAge'].apply(age_group)\ndf['Severity'] = df['Severity'].str.title()\ndf['Outcome'] = df['Outcome'].str.title()\n\n# Descriptive seriousness flag for this exercise.\ndf['SeriousReport'] = (\n    df['Severity'].isin(['Severe', 'Life-Threatening']) |\n    df['Outcome'].eq('Hospitalized')\n)\n\ndisplay(df[['SuspectedDrug','AdverseReaction','Severity','AgeGroup','Outcome','SeriousReport']])

## 4. Reporting frequency


In [ ]:
total_reports = len(df)\ndrug_frequency = df['SuspectedDrug'].value_counts().rename_axis('SuspectedDrug').reset_index(name='ReportCount')\nreaction_frequency = df['AdverseReaction'].value_counts().rename_axis('AdverseReaction').reset_index(name='ReportCount')\n\nprint('Total reports:', total_reports)\ndisplay(drug_frequency)\ndisplay(reaction_frequency)

## 5. Serious-report proportion


In [ ]:
serious_reports = int(df['SeriousReport'].sum())\nserious_proportion = serious_reports / total_reports if total_reports else 0\n\nprint(f'Serious reports: {serious_reports}/{total_reports}')\nprint(f'Serious-report proportion: {serious_proportion:.1%}')\ndisplay(df['SeriousReport'].value_counts().rename_axis('SeriousReport').to_frame('ReportCount'))

## 6. Visualizations


In [ ]:
drug_frequency.set_index('SuspectedDrug')['ReportCount'].sort_values().plot(kind='barh')\nplt.title('Reported Events by Suspected Drug')\nplt.xlabel('Number of Reports')\nplt.tight_layout()\nplt.show()

In [ ]:
reaction_frequency.set_index('AdverseReaction')['ReportCount'].sort_values().plot(kind='barh')\nplt.title('Most Frequently Reported Adverse Reactions')\nplt.xlabel('Number of Reports')\nplt.tight_layout()\nplt.show()

In [ ]:
df['Severity'].value_counts().plot(kind='bar')\nplt.title('Reports by Severity')\nplt.xlabel('Severity')\nplt.ylabel('Number of Reports')\nplt.xticks(rotation=30, ha='right')\nplt.tight_layout()\nplt.show()

### Drug–event combinations


In [ ]:
drug_event = (df.groupby(['SuspectedDrug','AdverseReaction'])\n                .size().reset_index(name='ReportCount')\n                .sort_values('ReportCount', ascending=False))\ndisplay(drug_event)\n\ntop = drug_event.head(15).copy()\ntop['Drug–Event'] = top['SuspectedDrug'] + ' — ' + top['AdverseReaction']\ntop.sort_values('ReportCount').plot(x='Drug–Event', y='ReportCount', kind='barh', legend=False)\nplt.title('High-Frequency Drug–Event Combinations')\nplt.xlabel('Number of Reports')\nplt.tight_layout()\nplt.show()

## 7. Interpretation

In this small synthetic sample, every suspected drug–reaction pair occurs once. Therefore, there is **no repeated high-frequency pair** in this dataset. The analysis demonstrates the workflow, but the sample is too small to support a strong signal claim.

The serious-report flag is descriptive only. It combines severe/life-threatening severity or hospitalization and should not be interpreted as a clinical seriousness adjudication.


## 8. Limitations and governance

- Reporting counts are not incidence rates; they depend on reporting behavior and exposure.
- A suspected-drug/adverse-event pair is not evidence that the drug caused the event.
- The dataset is synthetic and very small.
- No denominator such as number of exposed patients is available.
- No adjustment is made for age, indication, comorbidity, dose, duration, or concomitant medicines.
- Any potential signal should be reviewed by qualified pharmacovigilance/clinical experts before action.


## 9. Reproducibility

Place this notebook and `adverse_drug_reactions_sample.csv` in the same folder. Run all cells from top to bottom. The notebook uses only pandas, NumPy, and Matplotlib.
